In [1]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.stats import norm
import ipywidgets as widgets
from IPython.display import display

##### Note: Every price path SOlution in here will use the exact discretization instead of euler linear approximation
## Geometric Brownian Motion:

$dS_t​=μS_t​dt+σS_t​dW_t​$

consists of a trend and random movement

I went over this specific topic in another video already, so i dont want to fucking repeat everything.

The solution for GBM:

### $S_{t+Δt}​=S_t ​exp[(μ−\frac{σ^2​}{2})Δt+σΔtZ]$

In [2]:
class GBM:
    def __init__(self, drift, volatility, dt, initial_asset_price):
        self.drift = drift
        self.volatility = volatility
        self.dt = dt
        self.asset_prices = [initial_asset_price]
        self.current_asset_price = initial_asset_price

    def time_step(self):
        dW = np.random.normal(0, math.sqrt(self.dt))
        dS = self.drift * self.dt * self.current_asset_price + self.volatility * self.current_asset_price * dW
        
        self.asset_prices.append(self.current_asset_price + dS)
        self.current_asset_price = self.current_asset_price + dS
    
    def time_step_exact(self):
        z = np.random.normal()

        self.current_asset_price *= math.exp(
            (self.drift - 0.5 * self.volatility**2) * self.dt
            + self.volatility * math.sqrt(self.dt) * z
        )

        self.asset_prices.append(self.current_asset_price)


In [3]:

def plot_simple_gbm(drift=.2,volatility=.3,initial_asset_price=300):
    plt.figure(figsize=(10,5))
    for i in range(10):
        process= GBM(drift=drift,volatility=volatility,dt=1/365,initial_asset_price=initial_asset_price)
    
    
    
        tte = 1
        while(tte - process.dt) > 0:
            process.time_step()
            tte = tte - process.dt
        plt.plot(process.asset_prices)
        
    plt.xlabel("Time Steps")
    plt.ylabel("Asset Price")
    plt.title("SImple BS Process")

widgets.interact(
    plot_simple_gbm,
    drift=(-2,2,0.01),
    volatility=(0.0001, 2,0.001),
    initial_asset_price=(0.01,10000,0.01)
)

interactive(children=(FloatSlider(value=0.2, description='drift', max=2.0, min=-2.0, step=0.01), FloatSlider(v…

<function __main__.plot_simple_gbm(drift=0.2, volatility=0.3, initial_asset_price=300)>

# Price of a Call
 - We CAN use the closed-form formula, but unless you want to deal with Fourier Transforms and calc 3 shit, go ahead.
 - Instead i use Monte Carlo SImulation. For me this is just much simpler. Also, it works for exotics (asian, lookback, barrier options...)

Monte Carlo:

$C=e^{−rT}\frac{1}{N}​\sum^{N}_{i=1}​max(S_{T}^{(i)}​−K,0)​$ 


Or use whatever weird payoff function you need for pricing kuwaiti premium grade uranium-enriched oil exotic options

In [72]:


class MonteCarloSim:
    def __init__(self, process, call_strike, n_options, rfr, tte, barrier=100, exotics=False):
        self.stochastic_processes = []
        n_steps = round(tte / process.dt)  # valid when T is a whole number of time steps


        
        for _ in range(n_options):
            path = copy.deepcopy(process)
            ttei = tte
            while ttei > 0:
                path.time_step()
                ttei -= process.dt
            self.stochastic_processes.append(path)

        
        self.final_prices = []

        # barrier uo means up and out, di means down and in, etc.
        self.payoffs = {'vanilla':[], 'asian': [], 'fixed lookback':[], 'floating lookback':[], 
                        'barrier uo':[], 'barrier do':[], 'barrier ui':[], 'barrier di':[],
                       'binary':[]}
        self.prices = {}
        
        
        for stochastic_process in self.stochastic_processes:
            final_price = stochastic_process.asset_prices[len(stochastic_process.asset_prices)-1]
            self.final_prices.append(final_price)

            # vanilla payoff
            payoff = max(final_price - call_strike, 0)
            self.payoffs['vanilla'].append(payoff)

            if exotics:

                # asian payoff
                avg_spot_price = np.mean(stochastic_process.asset_prices)
                payoff = max(avg_spot_price - call_strike, 0)
                self.payoffs['asian'].append(payoff)
    
                # lookback payoff
                min_spot_price = np.min(stochastic_process.asset_prices)
                max_spot_price = np.max(stochastic_process.asset_prices)
                #fixed
                payoff = max(max_spot_price - call_strike, 0)
                self.payoffs['fixed lookback'].append(payoff)
                #floating
                payoff = max(final_price - min_spot_price, 0) # ??? this doesnt consider the call's strike price parameter?
                self.payoffs['floating lookback'].append(payoff)
    
                # barrier options payoff
                '''
                1. Up-and-out: spot price starts below the barrier level and has to move up for the option to be knocked out.
                2. Down-and-out: spot price starts above the barrier level and has to move down for the option to become null and void.
                3. Up-and-in: spot price starts below the barrier level and has to move up for the option to become activated.
                4. Down-and-in: spot price starts above the barrier level and has to move down for the option to become activated.
                '''
                if max_spot_price > barrier:
                    payoff_uo = 0
                else:
                    payoff_uo = max(final_price - call_strike, 0)
                if max_spot_price > barrier:
                    payoff_ui = max(final_price - call_strike, 0)
                else:
                    payoff_ui = 0
                if min_spot_price < barrier:
                    payoff_do = 0
                else:
                    payoff_do = max(final_price - call_strike, 0)
                if min_spot_price < barrier:
                    payoff_di = max(final_price - call_strike, 0)
                else:
                    payoff_di = 0
                
                self.payoffs['barrier uo'].append(payoff_uo)
                self.payoffs['barrier ui'].append(payoff_ui)
                self.payoffs['barrier do'].append(payoff_do)
                self.payoffs['barrier di'].append(payoff_di)
    
                
                # binary options
                if final_price > call_strike:
                    payoff = 100
                else: payoff = 0
                self.payoffs['binary'].append(payoff)
            
            
        discount = math.exp(-rfr * tte)
        for name, payoff_list in self.payoffs.items():
            #print(name, np.mean(payoff_list) * discount)
            self.prices[name] = np.mean(payoff_list) * discount


### Test Monte Carlo with simple GBM

In [41]:
import yfinance as yf
asset = yf.Ticker("SPY")
SPOT_PRICE = asset.fast_info.last_price
expirations = asset.options
day = 5
print(expirations[day])
calls = asset.option_chain(expirations[day]).calls
calls.sample(5)

2026-10-22


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency
10,SPY261022C00760000,2026-10-15 15:06:32+00:00,760.0,12.95,13.05,13.23,0.179999,1.409549,55.0,65.0,0.158639,True,REGULAR,USD
49,SPY261022C00799000,2026-10-15 14:17:11+00:00,799.0,0.05,0.06,0.07,0.000000,0.000000,2.0,117.0,0.114755,False,REGULAR,USD
32,SPY261022C00782000,2026-10-15 15:13:55+00:00,782.0,1.06,1.04,1.07,0.100000,10.416663,203.0,518.0,0.107126,False,REGULAR,USD
60,SPY261022C00815000,2026-10-15 15:07:12+00:00,815.0,0.02,0.02,0.03,0.000000,0.000000,882.0,423.0,0.150399,False,REGULAR,USD
37,SPY261022C00787000,2026-10-15 15:15:50+00:00,787.0,0.45,0.42,0.43,0.070000,18.421051,248.0,455.0,0.104623,False,REGULAR,USD


In [75]:
c = calls[calls['inTheMoney'] ==True].sample().iloc[0]
c

contractSymbol              SPY261022C00735000
lastTradeDate        2026-10-15 15:16:16+00:00
strike                                   735.0
lastPrice                                 38.3
bid                                      36.02
ask                                      37.33
change                                    -4.0
percentChange                        -9.456264
volume                                    27.0
openInterest                               1.0
impliedVolatility                      0.29261
inTheMoney                                True
contractSize                           REGULAR
currency                                   USD
Name: 3, dtype: object

In [80]:
test = MonteCarloSim(process=GBM(drift=0.08, volatility=c['impliedVolatility'], dt=1/365, initial_asset_price=SPOT_PRICE),
                     call_strike=c['strike'], n_options=1000,  rfr=0.04, tte=7/365, barrier=800, exotics=False)

In [81]:
test.prices

{'vanilla': 36.15753047233665,
 'asian': nan,
 'fixed lookback': nan,
 'floating lookback': nan,
 'barrier uo': nan,
 'barrier do': nan,
 'barrier ui': nan,
 'barrier di': nan,
 'binary': nan}